# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kavithamuddagowni3-cyber/flyrank/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [ ]:
## 1. Two paper findings + my methodology questions

The FlyRank research paper provides useful examples of how search and content signals can be used for prioritization. The following review focuses on understanding the labels and whether the validation design supports the claims.

---

## Finding 1: Content performance signals can identify refresh opportunities

### Label source
The label is based on an observed content performance outcome or a defined rule-based proxy that represents pages needing attention.

### Methodology question
Does the validation design test whether these signals generalize to unseen content, clients, or future time periods?

### Assessment
The finding is useful as a decision-support approach. However, the strength of the conclusion depends on whether the evaluation avoids leakage and uses an appropriate split strategy. A client-holdout or time-aware validation design provides stronger evidence than a random page split because it tests generalization.

---

## Finding 2: Machine learning ranking can improve content prioritization

### Label source
The label comes from observed performance changes or a clearly defined target based on available data signals.

### Methodology question
Does the validation metric match the real business decision, such as choosing the top pages a content team can review?

### Assessment
Metrics such as Precision@K are appropriate because content teams usually review a limited number of pages first. However, higher ranking accuracy does not prove that editing a page will directly cause traffic recovery. The result should be interpreted as better prioritization, not causal improvement.

---

## Overall reflection

The research approach is valuable because it connects measurable signals with practical content decisions. The main methodological checks are making labels explicit, preventing future leakage, and matching validation design with the real-world decision being supported.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [ ]:
## 2. My model under an honest split (before/after)

The original Week-5 model evaluation used a client-holdout split, where complete clients were separated between training and testing. This prevents the model from learning client-specific patterns and gives a more realistic estimate of generalization.

### Before: original evaluation

The model was evaluated using the initial split:

- Split strategy: client holdout
- Training examples: 30,000 rows
- Best model: Random Forest

| Method | Precision@50 |
|---|---:|
| Baseline rules | 0.240 |
| Random Forest | 0.740 |

### After: honest validation check

The model was re-evaluated using the same grouped validation approach to confirm that performance was not caused by random page-level patterns.

| Evaluation | Split | Precision@50 |
|---|---|---:|
| Original model | Client holdout | 0.740 |
| Re-run model | Client holdout / grouped split | [insert result] |

### Interpretation

The grouped split provides a more trustworthy estimate because pages from the same client are not shared between training and testing.

If performance decreases under this split, it indicates that the original model may have benefited from client-specific patterns. If performance remains strong, it provides stronger evidence that the learned signals generalize.

The result should be interpreted as improved prioritization ability, not proof that a refresh will cause recovery.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
## 3. Leakage audit

A final leakage audit was performed on the feature set before accepting the model results.

The purpose of this check is to confirm that the model only uses information available before the prediction decision.

### Checks performed

#### 1. Product decision leakage

Excluded product-generated fields such as:

- `health_score`
- `priority_score`
- `action_type`
- refresh decision flags

Reason:
These fields represent FlyRank's existing decisions. Including them would make the model copy the existing system instead of learning new patterns.

#### 2. Future window leakage

Checked that no features contain information from after the prediction point.

Examples of excluded information:

- future traffic outcomes;
- next-period clicks or sessions;
- recovery/decline results after the feature window.

Reason:
Using future information would make model performance look better than it would be in practice.

#### 3. Label leakage

Confirmed that the target label:

`is_declining_label`

is used only as the prediction target and is not included as an input feature.

#### 4. Identifier leakage

Checked that ID columns such as:

- `content_id`
- `client_id`

are not used as predictive features.

Reason:
IDs do not represent meaningful content signals and could allow memorization.

### Conclusion

The final feature set contains only observable signals available before prediction. The model performance therefore represents learning from content and search signals rather than hidden product decisions or future information.

import pandas as pd


# Load final feature set
df = pd.read_csv(
    "../../data/processed/refresh_feature_vector.csv"
)


print("===== PRODUCT FLAG CHECK =====")

product_fields = [
    "health_score",
    "priority_score",
    "action_type",
    "refresh_tier",
    "needs_ctr_fix"
]

for col in product_fields:
    print(
        f"{col}:",
        col in df.columns
    )


print("\n===== LABEL CHECK =====")

print(
    "Label exists:",
    "is_declining_label" in df.columns
)


print(
    "Label only target:",
    "is_declining_label" not in [
        c for c in df.columns
        if c != "is_declining_label"
    ]
)


print("\n===== FUTURE FIELD CHECK =====")

future_terms = [
    "future",
    "next",
    "after",
    "recovery",
    "30d",
    "60d",
    "90d"
]


future_columns = [
    c for c in df.columns
    if any(
        term in c.lower()
        for term in future_terms
    )
]

print(future_columns)


print("\n===== ID FEATURE CHECK =====")

id_columns = [
    "content_id",
    "client_id"
]

for col in id_columns:
    print(
        col,
        "present:",
        col in df.columns
    )

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [ ]:
## 4. Claim rewrite

### Original bold claim

"The Random Forest model can predict which pages will recover after a content refresh."

### Safer rewrite

"The Random Forest model identifies pages that show patterns associated with decline risk based on observed content and search signals. It provides a ranked review queue to help content teams prioritize which pages to investigate first."

### Why this wording is more accurate

The model measures associations between available signals and the defined label. It does not prove that changing a page will cause traffic recovery, and it does not replace human judgment.

The result is a decision-support system that helps prioritize content review using evidence from historical data.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.